# U.S. County Population by Census Year via IPUMS NHGIS API

This notebook retrieves historical total-population counts for every U.S. county
from the [IPUMS NHGIS](https://www.nhgis.org/) data collection using the
[IPUMS API](https://developer.ipums.org/docs/v2/apiprogram/) and the
[`ipumspy`](https://ipumspy.readthedocs.io/) Python client.

**What the notebook does:**

1. Connects to the NHGIS API and inspects metadata for the time-series table we need.
2. Submits an extract request for county-level total population across all decennial censuses.
3. Downloads the resulting data and loads it into a Pandas DataFrame.
4. Reshapes the data into a wide table: one row per county, one column per census year.
5. Exports the table to CSV.
6. Produces a time-series chart of the aggregate U.S. population.

### Prerequisites

- An [IPUMS account](https://www.nhgis.org/) registered for NHGIS access.
- An API key from <https://account.ipums.org/api_keys>.
- The key stored in the environment variable `IPUMS_API_KEY`.

```bash
pip install ipumspy pandas matplotlib
export IPUMS_API_KEY="your-key-here"
```

### Data source

We use **NHGIS Time Series Table A00** (*Persons: Total*) with **nominal**
geographic integration. This table links comparable total-population counts
across decennial censuses (1790–2020) at the county level. Under nominal
integration each row maps to a county as defined at the time of each census,
and a cell is non-empty only for censuses in which that county existed.

---
## 1. Imports and configuration

In [ ]:
import os
import re
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from ipumspy import (
    IpumsApiClient,
    AggregateDataExtract,
    TimeSeriesTable,
)
from ipumspy.api.metadata import TimeSeriesTableMetadata

# Directories for downloaded data and outputs
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZIP_PATH = DATA_DIR / "nhgis_extract.zip"
CSV_OUTPUT = DATA_DIR / "county_population_by_census_year.csv"

---
## 2. Connect to the IPUMS API

We read the API key from the `IPUMS_API_KEY` environment variable and
create an `IpumsApiClient` instance, which handles authentication,
retries, and rate limiting.

In [ ]:
api_key = os.environ.get("IPUMS_API_KEY")
if not api_key:
    raise RuntimeError(
        "Please set the IPUMS_API_KEY environment variable.\n"
        "Get a key at https://account.ipums.org/api_keys"
    )

client = IpumsApiClient(api_key)
print("API client ready.")

---
## 3. Inspect metadata for Time Series Table A00

Before submitting an extract we can query the API for metadata about
table A00 to confirm its description, available geographic levels, and
the census years it covers.

In [ ]:
meta = client.get_metadata(TimeSeriesTableMetadata("A00"))

print(f"Table:        A00")
print(f"Description:  {meta.description}")
print(f"Integration:  {getattr(meta, 'geographic_integration', 'nominal')}")
print(f"Geog levels:  {meta.geog_levels}")

years = getattr(meta, "years", None) or []
if years:
    print(f"Years:        {years}")

---
## 4. Define and submit the NHGIS extract

We request time-series table **A00** at the **county** geographic level.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `collection` | `"nhgis"` | NHGIS aggregate data collection |
| `time_series_tables` | `TimeSeriesTable("A00", geog_levels=["county"])` | Total population, county level, all available years |
| `data_format` | `"csv_header"` | CSV with a descriptive second header row |
| `tst_layout` | `"time_by_column_layout"` | Each census year becomes its own column |

In [ ]:
extract = AggregateDataExtract(
    collection="nhgis",
    description="County total population, time series A00 (nominal), all years",
    time_series_tables=[
        TimeSeriesTable("A00", geog_levels=["county"]),
    ],
    data_format="csv_header",
    tst_layout="time_by_column_layout",
)

print("Submitting extract...")
submitted = client.submit_extract(extract)
print(f"Extract submitted  \u2014  ID: {submitted.extract_id}")

---
## 5. Wait for the extract to complete and download it

The API processes the extract on IPUMS servers. `wait_for_extract` polls
with exponential back-off (starting at 1 s, capping at 300 s) until the
extract is ready, then we download the ZIP archive.

In [ ]:
print("Waiting for extract to complete (this may take a few minutes)...")
client.wait_for_extract(submitted)
print("Extract complete.")

client.download_extract(submitted, download_dir=str(DATA_DIR))
print(f"Downloaded to {DATA_DIR}/")

# Locate the downloaded zip file
zip_files = list(DATA_DIR.glob("nhgis*.zip"))
if zip_files:
    ZIP_PATH = zip_files[0]
    print(f"Using: {ZIP_PATH}")
else:
    raise FileNotFoundError(f"No NHGIS zip found in {DATA_DIR}")

---
## 6. Load the CSV from the downloaded ZIP

NHGIS packages its data inside the ZIP under a `*_csv/` sub-folder.
Because we chose `csv_header`, the file has a descriptive second row
that we skip when reading into Pandas.

In [ ]:
def load_nhgis_csv(zip_path: Path) -> pd.DataFrame:
    """Read the first county-level CSV from an NHGIS ZIP archive."""
    with zipfile.ZipFile(zip_path, "r") as zf:
        csv_files = [
            name for name in zf.namelist()
            if name.lower().endswith(".csv") and "_csv/" in name.replace("\\", "/")
        ]
        if not csv_files:
            raise FileNotFoundError("No CSV found inside the NHGIS zip.")

        # Prefer a file with 'county' in the name
        county_files = [f for f in csv_files if "county" in f.lower()]
        chosen = (county_files or csv_files)[0]

        with zf.open(chosen) as f:
            df = pd.read_csv(f, header=0, skiprows=[1], low_memory=False)

    return df


raw = load_nhgis_csv(ZIP_PATH)
print(f"Loaded {len(raw)} rows, {len(raw.columns)} columns")
raw.head()

---
## 7. Reshape into wide format: one row per county, census years as columns

NHGIS names its data columns with a table/series prefix followed by a
4-digit year (e.g. `A00AA1790`, `A00AA2020`). We rename those columns
to just the year string, keep the identifier columns (GISJOIN, state,
county name, etc.), and filter to *current* counties by requiring a
non-empty 2020 population value.

In [ ]:
YEAR_PATTERN = re.compile(r"^.+?(\d{4})$")

id_cols = []
year_rename = {}  # original column name -> year string

for col in raw.columns:
    m = YEAR_PATTERN.match(str(col).strip())
    if m and 1790 <= int(m.group(1)) <= 2100:
        year_rename[str(col).strip()] = m.group(1)
    else:
        id_cols.append(str(col).strip())

year_cols = sorted(year_rename.values(), key=int)
print(f"Identifier columns: {id_cols}")
print(f"Census year columns: {year_cols}")

In [ ]:
# Rename data columns from e.g. A00AA2020 -> 2020
wide = raw.rename(columns=year_rename)

# Keep only counties that have a value for the 2020 census
# (this serves as a proxy for "current" counties)
if "2020" in wide.columns:
    wide = wide[wide["2020"].notna() & (wide["2020"].astype(str).str.strip() != "")].copy()
    print(f"Filtered to {len(wide)} current counties (non-null 2020 population)")

# Convert year columns to numeric
for yr in year_cols:
    if yr in wide.columns:
        wide[yr] = pd.to_numeric(wide[yr], errors="coerce")

# Final table: identifiers + year columns
county_pop = wide[id_cols + [y for y in year_cols if y in wide.columns]].copy()
county_pop.head(10)

---
## 8. Export to CSV

In [ ]:
county_pop.to_csv(CSV_OUTPUT, index=False)
print(f"Saved {CSV_OUTPUT}  ({county_pop.shape[0]} rows x {county_pop.shape[1]} cols)")

---
## 9. Graphical summary: U.S. total population over time

We sum population across all counties for each census year and plot the
result as a time series. Missing values are ignored in the sum.

In [ ]:
present_years = [y for y in year_cols if y in county_pop.columns]
totals = county_pop[present_years].sum(skipna=True)
years_int = [int(y) for y in totals.index]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years_int, totals.values, marker="o", linewidth=2, markersize=5, color="#2563eb")
ax.fill_between(years_int, totals.values, alpha=0.1, color="#2563eb")
ax.set_title("U.S. Total Population by Decennial Census (NHGIS A00, County Sum)", fontsize=14)
ax.set_xlabel("Census Year", fontsize=12)
ax.set_ylabel("Population", fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = DATA_DIR / "us_total_population_timeseries.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved figure to {fig_path}")

---
## 10. Bonus: top 10 most populous counties over time

In [ ]:
# Identify the 10 most populous counties as of the latest available year
latest_year = present_years[-1]
top10 = county_pop.nlargest(10, latest_year)

# Build a label from state + county name columns (NHGIS typically has STATE and COUNTY)
name_col = next((c for c in id_cols if "county" in c.lower() and "code" not in c.lower()), None)
state_col = next((c for c in id_cols if "state" in c.lower() and "code" not in c.lower()), None)

fig, ax = plt.subplots(figsize=(14, 6))
for _, row in top10.iterrows():
    label_parts = []
    if name_col and pd.notna(row.get(name_col)):
        label_parts.append(str(row[name_col]).strip())
    if state_col and pd.notna(row.get(state_col)):
        label_parts.append(str(row[state_col]).strip())
    label = ", ".join(label_parts) if label_parts else str(row.get("GISJOIN", ""))

    vals = row[present_years].values.astype(float)
    ax.plot(years_int, vals, marker=".", linewidth=1.5, label=label)

ax.set_title(f"Top 10 Most Populous Counties (by {latest_year} census)", fontsize=14)
ax.set_xlabel("Census Year", fontsize=12)
ax.set_ylabel("Population", fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig_path2 = DATA_DIR / "top10_counties_timeseries.png"
fig.savefig(fig_path2, dpi=150)
plt.show()
print(f"Saved figure to {fig_path2}")

---

## Summary

| Output file | Description |
|---|---|
| `data/county_population_by_census_year.csv` | Wide table — one row per current county, columns for each decennial census |
| `data/us_total_population_timeseries.png` | Aggregate U.S. population time series |
| `data/top10_counties_timeseries.png` | Population trends for the 10 largest counties |

**Data source:** IPUMS NHGIS, University of Minnesota, [www.nhgis.org](https://www.nhgis.org/).

**Citation:** Steven Manson, Jonathan Schroeder, David Van Riper, Tracy Kugler, and Steven Ruggles.
IPUMS National Historical Geographic Information System: Version 18.0 [dataset].
Minneapolis, MN: IPUMS. http://doi.org/10.18128/D050.V18.0